## Objective

The goal of this notebook is to explore the structure of the ClinicalTrials.gov API response and identify the entities that should be represented in a relational database.

This notebook is exploratory only.

The implementation of the ETL pipeline will be developed inside the `src/` package.# ClinicalTrials.gov API - Data Modeling Exploration

### Initializations

In [2]:
import json
from pprint import pprint

from src.extract import fetch_studies

## Retrieve a sample study

In [3]:
studies = fetch_studies(max_studies=1)

study = studies[0]

## Top-level response structure

Let's inspect the first level of the JSON document.

In [5]:
study.keys()

dict_keys(['protocolSection', 'derivedSection', 'hasResults'])

### Protocol Section

In [6]:
protocol = study["protocolSection"]
protocol.keys()

dict_keys(['identificationModule', 'statusModule', 'sponsorCollaboratorsModule', 'oversightModule', 'descriptionModule', 'conditionsModule', 'designModule', 'armsInterventionsModule', 'outcomesModule', 'eligibilityModule', 'contactsLocationsModule'])

#### Identification Module

In [15]:
protocol_modules = protocol.keys()

for module_name in protocol_modules:
    print(f"\n{'=' * 80}")
    print(module_name)
    print(f"{'=' * 80}")

    pprint(protocol[module_name])


identificationModule
{'briefTitle': 'Multimedia Educational Program for Patients With Early-Stage '
               'Prostate Cancer or Breast Cancer',
 'nctId': 'NCT00830635',
 'officialTitle': 'Cancer Information Service Research Consortium: Years '
                  '2006-2011 Program Narrative and Overview',
 'orgStudyIdInfo': {'id': '08-0498'},
 'organization': {'class': 'OTHER',
                  'fullName': 'University of Colorado, Denver'},
 'secondaryIdInfos': [{'id': 'AMCCRC-08-0498'}]}

statusModule
{'completionDateStruct': {'date': '2013-12', 'type': 'ACTUAL'},
 'expandedAccessInfo': {'hasExpandedAccess': False},
 'lastUpdatePostDateStruct': {'date': '2015-06-16', 'type': 'ESTIMATED'},
 'lastUpdateSubmitDate': '2015-06-12',
 'overallStatus': 'COMPLETED',
 'primaryCompletionDateStruct': {'date': '2013-04', 'type': 'ACTUAL'},
 'startDateStruct': {'date': '2008-09'},
 'statusVerifiedDate': '2015-06',
 'studyFirstPostDateStruct': {'date': '2009-01-28', 'type': 'ESTIMATED'},
 's

## Module Selection

After exploring all the modules available in the `protocolSection`, the following decisions were made for the first version of the data model.

### identificationModule ✅

Contains the study identifier (`nctId`) and descriptive information. This module will become the core `studies` table.

**Used for:**
- Identifying each clinical trial.
- Providing study titles and metadata.

---

### statusModule ✅

Provides recruitment status and key study dates.

**Used for:**
- Timeline analysis of study durations.
- Study completion status.
- Intervention completion rate analysis.

---

### sponsorCollaboratorsModule ❌

Contains sponsor and collaborator information. It's not required to answer the analytical questions defined in this challenge.

---

### oversightModule ❌

Contains regulatory metadata. It's not required to answer the analytical questions defined in this challenge.

---

### descriptionModule ❌

Contains long free-text descriptions. These fields are useful for documentation but are not needed for the analytical queries.

---

### conditionsModule ✅

Contains the medical conditions studied in each clinical trial.

**Used for:**
- Identifying the most common conditions being studied.

---

### designModule ✅

Contains study design information, including study type, phase and enrollment.

**Used for:**
- Number of trials by study type and phase.
- Patient enrollment analysis.

---

### armsInterventionsModule ✅

Contains all interventions evaluated in the study.

**Used for:**
- Intervention analysis.
- Identifying interventions with the highest completion rates.

---

### outcomesModule ❌

These data are not required to answer the analytical questions requested in the challenge.

---

### eligibilityModule ❌

These data are not required to answer the analytical questions requested in the challenge.

---

### contactsLocationsModule ✅

Contains study locations and participating facilities.

**Used for:**
- Geographic distribution of clinical trials.

---

# Analytical Questions Covered

The selected modules allow the pipeline to answer all the analytical questions proposed in the challenge:

| Analytical Question | Module(s) |
|---------------------|-----------|
| How many trials by study type and phase? | `identificationModule`, `designModule` |
| What are the most common conditions being studied? | `conditionsModule` |
| Which interventions have the highest completion rates? | `armsInterventionsModule`, `statusModule` |
| Geographic distribution of clinical trials | `contactsLocationsModule` |
| Timeline analysis of study durations | `statusModule` |
| Patient enrollment analysis | `designModule` (`enrollmentInfo`) |

## Derived Section

The `derivedSection` contains metadata enriched by ClinicalTrials.gov rather than original study data.

It is out of scope for the first version of this pipeline and will not be considered.

## hasResults

`hasResults` indicates whether a study has published results.

It is out of scope for the first version of this pipeline.